In [ ]:
%pip install tensorflow==2.15.0 tensorflow-privacy==0.8.12 --no-deps

%pip install pandas shap matplotlib numpy

In [ ]:
import sys
import subprocess

def install_and_check():
    print(" Начинаем комплексную настройку окружения...")

    # 1. Основные тяжелые библиотеки
    base_packages = [
        "tensorflow==2.15.0",
        "pandas",
        "shap",
        "matplotlib",
        "numpy",
        "packaging"
    ]
    
    # 2. Математические зависимости для Privacy (ставим их ДО самой библиотеки)
    privacy_deps = [
        "dp-accounting==0.4.3",
        "tensorflow-probability==0.22.1",
        "dm-tree==0.1.8",
        "immutabledict",
        "absl-py",
        "attrs"
    ]

    print("\n Шаг 1: Установка базового стека и зависимостей...")
    all_deps = base_packages + privacy_deps
    subprocess.run([sys.executable, "-m", "pip", "install", *all_deps])

    print("\n Шаг 2: Установка TensorFlow Privacy (режим изоляции)...")
    # Ставим без зависимостей, чтобы pip не пытался скачать конфликтующий tf-models-official
    subprocess.run([sys.executable, "-m", "pip", "install", "tensorflow-privacy==0.8.12", "--no-deps"])

    print("\n Шаг 3: Финальная проверка импортов...")
    try:
        import tensorflow as tf
        import tensorflow_privacy as tfp
        import pandas as pd
        import dp_accounting
        import shap
        
        print(f"\n✅ ВСЁ ГОТОВО!")
        print(f"--- TensorFlow: {tf.__version__}")
        print(f"--- TF Privacy: Успешно загружен")
        print(f"--- DP Accounting: Доступен")
        return True
    except ImportError as e:
        print(f"\n❌ Ошибка при проверке: {e}")
        return False

# Запуск
if install_and_check():
    print("\nОкружение настроено. (Restart Kernel)")

In [1]:
import pandas as pd
import tensorflow as tf
import tensorflow_privacy as tfp
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate
from tensorflow.keras.models import Model
import numpy as np
import time
import gc

In [2]:
# Загрузка данных
print("Загружаем данные...")
train_path = r'C:\Users\slava\OneDrive\Рабочий стол\PyDissertation\Criteo_x1\train.csv'
test_path  = r'C:\Users\slava\OneDrive\Рабочий стол\PyDissertation\Criteo_x1\test.csv'
valid_path = r'C:\Users\slava\OneDrive\Рабочий стол\PyDissertation\Criteo_x1\valid.csv'

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)
df_valid = pd.read_csv(valid_path)

Загружаем данные...


In [3]:
# Проверяем, что всё загрузилось успешно
print("Размер тренировочной выборки (строки, колонки):", df_train.shape)
print("Размер тестовой выборки:", df_test.shape)
print("Размер валидационной выборки:", df_valid.shape)

# Снимаем ограничение на количество отображаемых колонок
pd.set_option('display.max_columns', None)

# Смотрим первые 5 строк тренировочного датасета
df_train.head()

Размер тренировочной выборки (строки, колонки): (33003326, 40)
Размер тестовой выборки: (4587167, 40)
Размер валидационной выборки: (8250124, 40)


,label,I1,I2,I3,I4,I5,I6,I7,I8,I9,I10,I11,I12,I13,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,C15,C16,C17,C18,C19,C20,C21,C22,C23,C24,C25,C26
0,0,0.05,0.006633,0.05,0.00,0.021594,0.008,0.15,0.04,0.362,0.1,0.2,0.0,0.04,15,1484,2032,421036,664216,664522,666759,676735,677367,677738,733335,737432,1148460,1150514,1150756,1184372,1528982,1529013,1533925,1536019,1556876,1934144,1934164,1936312,2022803,2024738
1,0,0.10,0.004975,0.44,0.02,0.001594,0.016,0.02,0.04,0.008,0.1,0.1,0.0,0.08,15,1490,2042,415618,664216,664524,666676,676733,677367,678156,732765,737443,1147992,1150512,1150579,1163048,1528983,1529008,1533925,1536020,1536034,1934144,1934164,1934195,2022803,2022906
2,0,0.10,0.004975,0.01,0.28,0.011984,0.178,0.04,0.04,0.490,0.1,0.3,0.3,0.90,37,1522,2034,415607,664216,664522,664925,676733,677367,677371,732554,737435,1147748,1150513,1151267,1163039,1528988,1529304,1533924,1536018,1536025,1934145,1934164,1934181,2022801,2022897
3,0,0.00,1.000000,0.00,0.00,0.068625,0.000,0.00,0.00,0.000,0.0,0.0,0.0,0.00,15,1487,2032,415832,664216,664524,664614,676733,677367,677372,732086,737432,1147333,1150513,1150556,1163036,1528989,1529022,1533924,1536018,1536022,1934144,1934164,1934185,2022801,2022897
4,0,0.15,0.003317,0.00,0.00,0.000031,0.000,0.03,0.00,0.000,0.1,0.1,0.0,0.00,17,1504,2032,415612,664216,664525,665982,676733,677367,678333,732804,737472,1148026,1150514,1150892,1163052,1528989,1529288,1533924,1536018,1536038,1934144,1934163,1934180,2022801,2022897


In [4]:
df_test.head()

,label,I1,I2,I3,I4,I5,I6,I7,I8,I9,I10,I11,I12,I13,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,C15,C16,C17,C18,C19,C20,C21,C22,C23,C24,C25,C26
0,1,0.0,0.008292,0.11,0.10,0.160344,0.068,0.02,0.08,0.010,0.0,0.1,0.0,0.10,18,1479,2032,420661,664216,664521,664814,676748,677367,677662,732093,737432,1147338,1150514,1150550,1163036,1528983,1528994,1534050,1536021,1536022,1934144,1934163,1934311,2022806,2024736
1,1,0.0,0.134328,0.02,0.30,0.067359,0.170,0.04,0.36,0.460,0.0,0.3,0.0,0.30,15,1669,168320,530790,664217,664526,664929,676733,677367,677606,732155,910458,1147414,1150512,1151620,1318886,1528982,1529471,1533924,1536018,1707732,1934144,1934163,1935911,2022801,2022897
2,0,0.0,0.003317,0.00,0.00,0.077438,0.000,0.00,0.74,0.194,0.0,0.0,0.0,0.00,18,1508,2033,415607,664216,664524,665408,676733,677367,677371,733145,737433,1147419,1150518,1150585,1163037,1528986,1529049,1533924,1536018,1536023,1934144,1934169,1934181,2022801,2022897
3,1,1.0,0.533997,0.00,0.08,0.000078,0.008,0.89,0.80,0.176,0.3,0.4,1.0,0.08,14,1486,2208,415748,664216,664523,664568,676733,677367,677567,732126,737610,1147380,1150512,1150557,1163164,1528988,1529017,1533924,1536018,1536209,1934144,1934163,1934312,2022801,2022897
4,0,0.0,0.097844,0.02,0.00,0.000000,0.000,0.00,0.00,0.002,0.0,0.0,0.0,0.00,16,1497,2034,415607,664216,664523,664579,676734,677368,677371,732150,737435,1147335,1150512,1150612,1163039,1528990,1529030,1533924,1536018,1536025,1934144,1934167,1934181,2022801,2022897


In [5]:
df_valid.head()

,label,I1,I2,I3,I4,I5,I6,I7,I8,I9,I10,I11,I12,I13,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,C15,C16,C17,C18,C19,C20,C21,C22,C23,C24,C25,C26
0,0,0.00,0.006633,0.02,0.00,0.049500,0.000,0.00,0.02,0.004,0.0,0.0,0.0,0.00,30,1540,2082,415670,664218,664524,674803,676733,677367,677371,732721,737488,1147952,1150513,1150719,1163099,1528986,1529077,1533924,1536018,1536080,1934144,1934171,1934205,2022801,2022897
1,0,0.00,0.077944,0.04,0.16,0.297031,0.498,0.28,0.62,0.282,0.0,0.1,0.0,0.16,14,1496,2033,415607,664216,664522,666035,676733,677367,685051,733012,737433,1148010,1150512,1150764,1163037,1528982,1529141,1533924,1536018,1536023,1934144,1934163,1934181,2022801,2022897
2,0,0.00,0.014925,0.06,0.12,0.006578,0.218,0.01,0.14,0.214,0.0,0.1,0.0,0.12,14,1901,2032,415606,664217,664521,667284,676733,677367,677371,733187,737432,1148336,1150513,1152548,1163036,1528984,1530174,1533924,1536018,1536022,1934144,1934163,1934178,2022801,2022897
3,1,0.00,0.003317,0.00,0.00,0.022891,0.000,0.17,0.00,0.008,0.0,0.4,0.0,0.00,26,1475,2032,415614,664216,664524,664543,676735,677367,677391,732085,737432,1147332,1150513,1151648,1163043,1528982,1529023,1533924,1536018,1536029,1934144,1934163,1934180,2022801,2022897
4,0,0.35,0.174129,0.00,0.06,0.012188,0.030,0.07,0.30,0.030,0.1,0.1,0.0,0.06,40,1626,4768,419345,664217,664525,665930,676733,677367,678669,732871,740240,1147913,1150522,1157583,1166229,1528982,1531458,1533924,1536018,1538936,1934144,1934172,1936323,2022801,2022897


In [ ]:
# Выборка
print("Делаем выборку...")
SAMPLE_SIZE = 300000
df_train_sample = df_train.sample(n=SAMPLE_SIZE, random_state=42)
df_test_sample = df_test.sample(n=int(SAMPLE_SIZE * 0.2), random_state=42)
df_valid_sample = df_valid.sample(n=int(SAMPLE_SIZE * 0.2), random_state=42)

Делаем выборку...


In [ ]:
# Определение колонок
target = 'label'
dense_features = [f'I{i}' for i in range(1, 14)] 
sparse_features = [f'C{i}' for i in range(1, 27)] 

In [ ]:
# ПОДГОТОВКА ВХОДОВ ДЛЯ НЕЙРОСЕТИ
# Вход для числовых (Dense) признаков
dense_inputs = Input(shape=(len(dense_features),), name='dense_inputs')

sparse_inputs = {}
sparse_embeddings = []

print("Создаем слои Embedding для категорий...")
for feat in sparse_features:
    # Ищем максимальный индекс категории, чтобы задать размер словаря
    # Ищем по всем таблицам, чтобы избежать ошибки OutOfBounds
    vocab_size = int(max(df_train[feat].max(), df_valid[feat].max(), df_test[feat].max()) + 1)
    
    # Входной слой для конкретной категории
    inp = Input(shape=(1,), name=feat)
    sparse_inputs[feat] = inp
    
    # Embedding-слой (сжимаем категорию в вектор из 8 чисел)
    emb = Embedding(input_dim=vocab_size, output_dim=4)(inp)
    sparse_embeddings.append(Flatten()(emb))


Создаем слои Embedding для категорий...


In [ ]:
# WIDE ЧАСТЬ (Запоминание)
wide_output = dense_inputs

In [ ]:
# DEEP ЧАСТЬ (Обобщение)
# Объединяем эмбеддинги и числовые данные
deep_concat = Concatenate()(sparse_embeddings + [dense_inputs])
deep_layer = Dense(128, activation='relu')(deep_concat)
deep_layer = Dense(64, activation='relu')(deep_layer)
deep_layer = Dense(32, activation='relu')(deep_layer)

In [ ]:
# СЛИЯНИЕ WIDE И DEEP
final_concat = Concatenate()([wide_output, deep_layer])

In [ ]:
# Выходной слой (сигмоида для предсказания вероятности клика от 0 до 1)
output = Dense(1, activation='sigmoid', name='output')(final_concat)

In [ ]:
# СБОРКА И КОМПИЛЯЦИЯ МОДЕЛИ
model_inputs = [dense_inputs] + list(sparse_inputs.values())
model = Model(inputs=model_inputs, outputs=output)

In [ ]:
# Собираем модель. Используем базовый Adam (Differential Privacy добавим позже для сравнения)
model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['AUC'])

print("Архитектура Wide & Deep успешно создана!")
model.summary()

Архитектура Wide & Deep успешно создана!
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 C1 (InputLayer)             [(None, 1)]                  0         []                            
                                                                                                  
 C2 (InputLayer)             [(None, 1)]                  0         []                            
                                                                                                  
 C3 (InputLayer)             [(None, 1)]                  0         []                            
                                                                                                  
 C4 (InputLayer)             [(None, 1)]                  0         []                            
                                                     

In [ ]:
# Упаковка данных в словари
def get_keras_data_safe(df, dense_cols, sparse_cols):
    X = {}
    X['dense_inputs'] = df[dense_cols].values.astype(np.float32)
    for col in sparse_cols:
        X[col] = df[col].values.astype(np.int32)
    return X

In [ ]:
print("Упаковываем данные для Keras...")
X_train = get_keras_data_safe(df_train_sample, dense_features, sparse_features)
y_train = df_train_sample[target].values.astype(np.float32)

X_valid = get_keras_data_safe(df_valid_sample, dense_features, sparse_features)
y_valid = df_valid_sample[target].values.astype(np.float32)

X_test = get_keras_data_safe(df_test_sample, dense_features, sparse_features)
y_test = df_test_sample[target].values.astype(np.float32)

Упаковываем данные для Keras...


In [ ]:
# Очищаем исходные огромные датафреймы, чтобы освободить RAM перед тяжелым циклом
del df_train, df_test, df_valid, df_train_sample, df_test_sample, df_valid_sample
gc.collect()

print("ДАННЫЕ УСПЕШНО ПОДГОТОВЛЕНЫ И ЗАГРУЖЕНЫ В ПАМЯТЬ!")

ДАННЫЕ УСПЕШНО ПОДГОТОВЛЕНЫ И ЗАГРУЖЕНЫ В ПАМЯТЬ!


In [ ]:
# Настройки эксперимента
embedding_sizes = [2, 4, 8]
results_data = []

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', patience=1, restore_best_weights=True, mode='max'
)

def build_clean_model(emb_dim):
    dense_inputs = Input(shape=(len(dense_features),), name='dense_inputs')
    sparse_inputs = {}
    sparse_embeddings = []

    for feat in sparse_features:
        vocab_size = int(max(X_train[feat].max(), X_valid[feat].max(), X_test[feat].max()) + 1)
        inp = Input(shape=(1,), name=feat)
        sparse_inputs[feat] = inp
        emb = Embedding(input_dim=vocab_size, output_dim=emb_dim)(inp)
        sparse_embeddings.append(Flatten()(emb))

    wide_output = dense_inputs
    deep_concat = Concatenate()(sparse_embeddings + [dense_inputs])
    deep_layer = Dense(128, activation='relu')(deep_concat)
    deep_layer = Dense(64, activation='relu')(deep_layer)
    deep_layer = Dense(32, activation='relu')(deep_layer)

    final_concat = Concatenate()([wide_output, deep_layer])
    output = Dense(1, activation='sigmoid', name='output')(final_concat)

    model_inputs = [dense_inputs] + list(sparse_inputs.values())
    return Model(inputs=model_inputs, outputs=output)

In [ ]:
print("🚀 Запуск автоматизированного исследования Эмбеддингов (Ablation Study)...\n")
start_total_time = time.time()

for d in embedding_sizes:
    print(f"{'='*50}")
    print(f"ТЕСТИРОВАНИЕ РАЗМЕРНОСТИ EMBEDDING = {d}")
    print(f"{'='*50}\n")
    
    # --- БАЗОВАЯ МОДЕЛЬ ---
    print("-> 1. Обучение Базовой модели (Baseline)...")
    base_model = build_clean_model(d)
    base_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    base_model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
                   batch_size=1024, epochs=3, callbacks=[early_stop], verbose=0) 
    
    _, base_auc = base_model.evaluate(X_test, y_test, batch_size=1024, verbose=0)
    print(f"   Базовый AUC: {base_auc:.4f}")
    total_params = base_model.count_params()

    # Сохраняем веса базовой модели
    base_model.save_weights(f'baseline_model_dim{d}.h5')
    print(f"   [Сохранено] baseline_model_dim{d}.h5")
    
    # --- ПРИВАТНАЯ МОДЕЛЬ ---
    print("-> 2. Обучение Приватной модели (DP-SGD)...")
    dp_model = build_clean_model(d)
    dp_optimizer = tfp.DPKerasAdamOptimizer(
        l2_norm_clip=1.0, noise_multiplier=0.1, num_microbatches=1, learning_rate=0.001
    )
    loss = tf.keras.losses.BinaryCrossentropy(reduction=tf.keras.losses.Reduction.NONE)
    dp_model.compile(optimizer=dp_optimizer, loss=loss, metrics=['AUC'])
    
    dp_model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
                 batch_size=1024, epochs=3, verbose=0)
    
    _, dp_auc = dp_model.evaluate(X_test, y_test, batch_size=1024, verbose=0)
    print(f"   Приватный AUC: {dp_auc:.4f}")
    print(f"   Падение: {(base_auc - dp_auc)*100:.2f}%\n")

    # Сохраняем веса приватной модели
    dp_model.save_weights(f'dp_model_dim{d}.h5')
    print(f"   [Сохранено] dp_model_dim{d}.h5\n")
    
    # Запись результатов в список
    results_data.append({
        'Размер Эмбеддинга (d)': d,
        'Кол-во параметров': f"{total_params / 1e6:.1f} млн",
        'Базовый AUC': round(base_auc, 4),
        'Приватный AUC (DP)': round(dp_auc, 4),
        'Падение точности': f"{(base_auc - dp_auc):.4f}",
        'В процентах': f"{(base_auc - dp_auc)/base_auc * 100:.2f}%"
    })
    
    del base_model
    del dp_model
    tf.keras.backend.clear_session()
    gc.collect()

print(f"\n✅ Эксперимент завершен! Общее время: {(time.time() - start_total_time)/60:.1f} минут.\n")

# Сохраняем итоговую таблицу в CSV
df_results = pd.DataFrame(results_data)
df_results.to_csv('ablation_results_embeddings.csv', index=False)
print("💾 Таблица с результатами сохранена в файл 'ablation_results_embeddings.csv'")

display(df_results)

🚀 Запуск автоматизированного исследования Эмбеддингов (Ablation Study)...

ТЕСТИРОВАНИЕ РАЗМЕРНОСТИ EMBEDDING = 2

-> 1. Обучение Базовой модели (Baseline)...


   Базовый AUC: 0.7669
   [Сохранено] baseline_model_dim2.h5
-> 2. Обучение Приватной модели (DP-SGD)...
   Приватный AUC: 0.6431
   Падение: 12.38%

   [Сохранено] dp_model_dim2.h5

ТЕСТИРОВАНИЕ РАЗМЕРНОСТИ EMBEDDING = 4

-> 1. Обучение Базовой модели (Baseline)...
   Базовый AUC: 0.7673
   [Сохранено] baseline_model_dim4.h5
-> 2. Обучение Приватной модели (DP-SGD)...
   Приватный AUC: 0.6676
   Падение: 9.98%

   [Сохранено] dp_model_dim4.h5

ТЕСТИРОВАНИЕ РАЗМЕРНОСТИ EMBEDDING = 8

-> 1. Обучение Базовой модели (Baseline)...
   Базовый AUC: 0.7666
   [Сохранено] baseline_model_dim8.h5
-> 2. Обучение Приватной модели (DP-SGD)...
   Приватный AUC: 0.6655
   Падение: 10.11%

   [Сохранено] dp_model_dim8.h5


✅ Эксперимент завершен! Общее время: 77.8 минут.

💾 Таблица с результатами сохранена в файл 'ablation_results_embeddings.c

,Размер Эмбеддинга (d),Кол-во параметров,Базовый AUC,Приватный AUC (DP),Падение точности,В процентах
0,2,60.3 млн,0.7669,0.6431,0.1238,16.14%
1,4,120.5 млн,0.7673,0.6676,0.0998,13.00%
2,8,241.0 млн,0.7666,0.6655,0.1011,13.19%


In [ ]:
def calculate_epsilon(n_samples, batch_size, epochs, noise_multiplier, target_delta):
    """Рассчитывает Эпсилон на основе параметров обучения"""
    
    # 1. Считаем, сколько всего шагов сделает оптимизатор
    steps_per_epoch = n_samples // batch_size
    total_steps = steps_per_epoch * epochs
    
    # 2. Настраиваем математический аппарат DP-SGD
    accountant = dp_accounting.rdp.RdpAccountant()
    
    # Формируем событие: "Мы добавили шум к батчу с такой-то вероятностью"
    event = dp_accounting.dp_event.PoissonSampledDpEvent(
        sampling_probability=batch_size / n_samples,
        event=dp_accounting.dp_event.GaussianDpEvent(noise_multiplier)
    )
    
    # 3. Применяем событие для всех шагов обучения
    accountant.compose(event, total_steps)
    
    # 4. Получаем итоговый Эпсилон
    epsilon = accountant.get_epsilon(target_delta=target_delta)
    return epsilon

In [ ]:
# --- ПАРАМЕТРЫ ---
BATCH_SIZE = 1024
EPOCHS = 3
NOISE = 0.1
DELTA = 1e-5 # Стандартное значение для Criteo (должно быть меньше 1/N)

In [ ]:
print("РАСЧЕТ ПРИВАТНОГО БЮДЖЕТА (EPSILON)")
print("-" * 40)

# Считаем для текущего Ablation Study
eps_300k = calculate_epsilon(n_samples=300000, batch_size=BATCH_SIZE, 
                             epochs=EPOCHS, noise_multiplier=NOISE, target_delta=DELTA)
print(f"1. Для выборки 300 000 строк:")
print(f"   Эпсилон (ε) = {eps_300k:.2f}")

# Считаем для будущего Финального прогона
eps_1m = calculate_epsilon(n_samples=1000000, batch_size=BATCH_SIZE, 
                           epochs=EPOCHS, noise_multiplier=NOISE, target_delta=DELTA)
print(f"\n2. Для выборки 1 000 000 строк:")
print(f"   Эпсилон (ε) = {eps_1m:.2f}")
print("-" * 40)

РАСЧЕТ ПРИВАТНОГО БЮДЖЕТА (EPSILON)
----------------------------------------
1. Для выборки 300 000 строк:
   Эпсилон (ε) = 3483.84

2. Для выборки 1 000 000 строк:
   Эпсилон (ε) = 3554.00
----------------------------------------


In [ ]:
def find_required_noise(n_samples, batch_size, epochs, target_epsilon, target_delta):
    """
    Ищет необходимый уровень шума (noise_multiplier) для достижения
    заданного уровня приватности (target_epsilon).
    Использует бинарный поиск (Binary Search).
    """
    steps_per_epoch = n_samples // batch_size
    total_steps = steps_per_epoch * epochs
    
    # Настраиваем границы поиска шума (от 0.1 до 10.0)
    low_noise = 0.1
    high_noise = 10.0
    tolerance = 0.05 # Допустимая погрешность Эпсилона
    
    print(f"Ищем оптимальный шум для достижения ε = {target_epsilon}...")
    
    while high_noise - low_noise > 0.001:
        mid_noise = (low_noise + high_noise) / 2
        
        # Считаем Эпсилон для текущего шума
        accountant = dp_accounting.rdp.RdpAccountant()
        event = dp_accounting.dp_event.PoissonSampledDpEvent(
            sampling_probability=batch_size / n_samples,
            event=dp_accounting.dp_event.GaussianDpEvent(mid_noise)
        )
        accountant.compose(event, total_steps)
        current_epsilon = accountant.get_epsilon(target_delta=target_delta)
        
        # Сдвигаем границы бинарного поиска
        if current_epsilon > target_epsilon:
            # Эпсилон слишком большой -> Приватность слабая -> Нужен БОЛЬШЕ шум
            low_noise = mid_noise
        else:
            # Эпсилон маленький -> Приватность сильная -> Можно уменьшить шум
            high_noise = mid_noise
            
    return high_noise

In [ ]:
# Задаем список размеров выборки для тестирования
sizes_to_test = [100000, 200000, 300000, 400000,500000, 600000, 700000, 800000, 900000, 1000000]

BATCH_SIZE = 1024
EPOCHS = 3
TARGET_EPSILON = 10.0

In [ ]:
print("-" * 50)
print("ИССЛЕДОВАНИЕ МАСШТАБИРОВАНИЯ ПРИВАТНОСТИ")
print("-" * 50)

for n in sizes_to_test:
    # Дельта всегда должна быть чуть меньше, чем 1 / размер_выборки
    current_delta = 1 / (n * 2) 
    
    req_noise = find_required_noise(n, BATCH_SIZE, EPOCHS, TARGET_EPSILON, current_delta)
    
    print(f"Выборка: {n} строк | Эпох: {EPOCHS} | Батч: {BATCH_SIZE}")
    print(f"Целевой ε: {TARGET_EPSILON} | Дельта: {current_delta}")
    print(f"Требуемый шум (noise_multiplier): {req_noise:.3f}")
    print("-" * 50)

--------------------------------------------------
ИССЛЕДОВАНИЕ МАСШТАБИРОВАНИЯ ПРИВАТНОСТИ
--------------------------------------------------
Ищем оптимальный шум для достижения ε = 10.0...
Выборка: 100000 строк | Эпох: 3 | Батч: 1024
Целевой ε: 10.0 | Дельта: 5e-06
Требуемый шум (noise_multiplier): 0.517
--------------------------------------------------
Ищем оптимальный шум для достижения ε = 10.0...
Выборка: 200000 строк | Эпох: 3 | Батч: 1024
Целевой ε: 10.0 | Дельта: 2.5e-06
Требуемый шум (noise_multiplier): 0.488
--------------------------------------------------
Ищем оптимальный шум для достижения ε = 10.0...
Выборка: 300000 строк | Эпох: 3 | Батч: 1024
Целевой ε: 10.0 | Дельта: 1.6666666666666667e-06
Требуемый шум (noise_multiplier): 0.473
--------------------------------------------------
Ищем оптимальный шум для достижения ε = 10.0...
Выборка: 400000 строк | Эпох: 3 | Батч: 1024
Целевой ε: 10.0 | Дельта: 1.25e-06
Требуемый шум (noise_multiplier): 0.464
----------------------